# LTR Expansion Selection — XGBoost → ONNX

Train a classifier to select which expanded terms improve nDCG@5.

**Input**: CSV: `query_id,term_id,energy,df,cosine_sim,node_degree,avg_neighbor_weight,label`
**Output**: `model.onnx`

In [ ]:
!pip install -q xgboost onnxmltools onnx skl2onnx pandas scikit-learn

In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/models/ltr_expansion_v1"
DATA_PATH = "/kaggle/input/expansion-features/expansion_features.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows, {df['label'].mean():.2%} positive")
df.head()

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

FEATURES = ["energy", "df", "cosine_sim", "node_degree"]
TARGET = "label"

X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    objective="binary:logistic",
    tree_method="hist",
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=20)

probs = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, probs)
print(f"Validation AUC: {auc:.4f}")
print(classification_report(y_val, (probs > 0.5).astype(int)))

In [ ]:
import onnx
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType

initial_types = [("input", FloatTensorType([None, len(FEATURES)]))]
onnx_model = convert_xgboost(model, initial_types=initial_types)

onnx_path = os.path.join(OUTPUT_DIR, "model.onnx")
onnx.save(onnx_model, onnx_path)
print(f"Saved ONNX model to {onnx_path}")
print(f"Model size: {os.path.getsize(onnx_path) / 1024:.1f} KB")

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(onnx_path)
onnx_out = sess.run(None, {"input": X_val[:5]})
print("ONNX output shapes:", [o.shape for o in onnx_out])
print("Sample predictions:", onnx_out[1][:5])  # probabilities